# Day 7: Exploratory Data Analysis (EDA)

## Load the Dataset
Every EDA (Exploratory Data Analysis) workflow starts with getting your data into a DataFrame and taking a first look — here we use the Iris dataset, a classic beginner-friendly dataset of flower measurements bundled directly with scikit-learn, so there's no file to download. `load_iris(as_frame=True)` returns the data already as a pandas-friendly object; `.frame` gives you the DataFrame with feature columns plus a numeric `target` column (0, 1, 2 for each species), and the next line maps those numbers to their actual species names using `iris.target_names`, so you get a readable `species` column instead of just digits. `df.head()` then prints the first 5 rows, letting you sanity-check that the columns and species labels look correct before doing any real analysis.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris(as_frame=True)  # loads the Iris dataset as pandas objects instead of raw arrays
df = iris.frame  # DataFrame with 4 feature columns + numeric 'target' column
# map numeric target (0,1,2) to readable species names using target_names
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))
print(df.head())

## Structure Check
Before analyzing any dataset, you need to confirm its basic shape and health: how many rows and columns it has, what type each column is, and whether anything is missing — skipping this step is how bugs and bad conclusions creep in later. `df.shape` gives the (rows, columns) size, `df.info()` lists each column's data type and non-null count in one summary, and `df.isnull().sum()` counts missing values per column (summing the `True`/`False` results of `isnull()`, where `True` counts as 1). The output confirms this dataset has 150 rows and 6 columns, all columns are fully populated (150 non-null in `info()`), and `isnull().sum()` shows zero missing values everywhere — so no cleaning is needed before moving on.

In [ ]:
print(df.shape)              # (rows, columns)
print(df.info())             # column dtypes and non-null counts
print(df.isnull().sum())     # count of missing values per column

## Univariate Analysis
"Univariate" means looking at a single variable in isolation — checking its range, central tendency, and spread — which helps you spot skew, unusual scales, or potential outliers before comparing it to anything else. `.describe()` prints count, mean, standard deviation, min/max, and quartiles for a numeric column as text, while `sns.histplot(..., kde=True)` gives you the visual equivalent: a histogram of value counts with a smoothed density curve overlaid. Reading the output, the `describe()` numbers tell you the typical sepal length and how much it varies, and the histogram shows the same information visually — you should see values roughly clustered between 4 and 8 cm, without extreme outliers.

In [ ]:
print(df['sepal length (cm)'].describe())  # count, mean, std, min/max, quartiles
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(df['sepal length (cm)'], kde=True)  # visual distribution + smoothed density curve
plt.show()

## Bivariate Analysis
"Bivariate" analysis looks at two variables together to see if and how they relate — a natural next step after understanding each column individually, and often where the most interesting insights show up (e.g. "do longer sepals also tend to have longer petals?"). `sns.scatterplot(x=..., y=..., hue='species', data=df)` plots every row as a point positioned by its sepal and petal length, colored by species, so you can see both the overall relationship between the two measurements and whether that relationship differs across species. In the output, you should see the three species forming fairly distinct clusters, with petal length generally increasing alongside sepal length — a visual hint that these features are correlated and useful for telling species apart.

In [ ]:
# Color by species to see whether the sepal-vs-petal length relationship differs across species
sns.scatterplot(x='sepal length (cm)', y='petal length (cm)', hue='species', data=df)
plt.show()

## Correlation Analysis
Correlation analysis quantifies the relationships you might have spotted visually, giving a precise number (from -1 to +1) for how strongly each pair of numeric features moves together — this helps identify redundant features (highly correlated ones may carry duplicate information) before building a model. `df.drop(columns=[...]).corr()` first removes the non-feature columns (`target`, `species`, since correlation only makes sense between numeric measurements) and then computes the pairwise correlation matrix for what's left. In the output, values near 1.0 (like petal length vs. petal width at ~0.96) indicate features that move almost perfectly together, while values near 0 (like sepal length vs. sepal width) indicate little to no linear relationship.

In [ ]:
# Drop non-numeric/label columns first, then compute the pairwise correlation matrix
print(df.drop(columns=['target', 'species']).corr())

## Category-wise Summary
Once you know a categorical column (like `species`) is meaningful, the natural next question is how the numeric features differ across those categories — this is exactly the kind of comparison that later informs whether a category is a good predictor for classification. `df.groupby('species').mean(numeric_only=True)` groups all rows by species and computes the average of every numeric column within each group (`numeric_only=True` skips any non-numeric columns automatically). The output shows one row per species with its average measurements — notice how petal length and width increase steadily from setosa to versicolor to virginica, which is exactly the kind of pattern that makes these features useful for distinguishing species.

In [ ]:
# Average of every numeric column, computed separately for each species group
print(df.groupby('species').mean(numeric_only=True))